# Multi-task ViT — inference on one image per class

For each morphological task, pick **one example image per severity class** (`0`, `+`, `++`, `+++`) from the fold-0 test predictions, run the model, and compare GT vs prediction.

All default paths are on the **cluster** dataset mount. Override with environment variables if needed:
- `CAPILLAROSCOPY_ROOT`
- `NFC_CHECKPOINT`
- `NFC_PRED_CSV`
- `NFC_IMAGE_ROOT`

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display
from PIL import Image

ROOT = Path.cwd()
if not (ROOT / "Model").exists() and (ROOT.parent / "Model").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from infer import (
    SEVERITY,
    TASK_DISPLAY,
    TASKS,
    load_model,
    predict_one,
)

CLUSTER_BASE = Path(os.environ.get("CAPILLAROSCOPY_ROOT", "/cluster/dataset/medinfmk/capillaroscopy"))
CHECKPOINT = Path(
    os.environ.get(
        "NFC_CHECKPOINT",
        str(CLUSTER_BASE / "Nail-Imaging/multi-task/multiTask/0/pytorch_model.bin"),
    )
)
PRED_CSV = Path(
    os.environ.get(
        "NFC_PRED_CSV",
        str(CLUSTER_BASE / "Nail-Imaging/multi-task/image_ids_test-set_0.csv"),
    )
)
IMAGE_ROOT = Path(os.environ.get("NFC_IMAGE_ROOT", str(CLUSTER_BASE / "content/images")))
OUT_DIR = ROOT / "infer_examples_outputs"
OUT_DIR.mkdir(exist_ok=True)


def resolve_image_path(path: str) -> Path:
    """Map CSV image paths to IMAGE_ROOT by filename."""
    p = Path(path)
    if p.is_file():
        return p
    candidate = IMAGE_ROOT / p.name
    if candidate.is_file():
        return candidate
    return IMAGE_ROOT / p.name


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("checkpoint:", CHECKPOINT)
print("predictions CSV:", PRED_CSV)
print("image root:", IMAGE_ROOT)
print("output dir:", OUT_DIR.relative_to(ROOT) if OUT_DIR.is_relative_to(ROOT) else OUT_DIR.name)

In [ ]:
assert CHECKPOINT.is_file(), f"Checkpoint not found: {CHECKPOINT}"
assert PRED_CSV.is_file(), f"CSV not found: {PRED_CSV}"
assert IMAGE_ROOT.is_dir(), f"Image root not found: {IMAGE_ROOT}"

df = pd.read_csv(PRED_CSV, index_col=0)
df.index = df.index.map(lambda x: str(resolve_image_path(x)))
print(f"Loaded {len(df)} test images")
df.head(2)

In [ ]:
model = load_model(str(CHECKPOINT), device)
print("Model loaded.")

## Pick one image per severity class

For every task, take the first available test image whose **ground-truth** severity is `0`, `+`, `++`, or `+++`.

In [ ]:
def pick_one_per_class(frame: pd.DataFrame, task: str):
    chosen = {}
    for class_id in range(4):
        subset = frame[frame[task] == class_id]
        for path in subset.index:
            if Path(path).is_file():
                chosen[class_id] = path
                break
    return chosen


examples = {task: pick_one_per_class(df, task) for task in TASKS}
for task, mapping in examples.items():
    print(TASK_DISPLAY[task], {SEVERITY[k]: Path(v).name for k, v in mapping.items()})

## Run inference and plot

Figures are written under `infer_examples_outputs/`.

In [ ]:
def run_and_show(task: str, examples_for_task: dict):
    class_ids = sorted(examples_for_task)
    n = len(class_ids)
    if n == 0:
        print(f"No examples for {task}")
        return

    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4.5))
    if n == 1:
        axes = [axes]

    rows = []
    for ax, class_id in zip(axes, class_ids):
        path = examples_for_task[class_id]
        gt = SEVERITY[class_id]
        if not Path(path).is_file():
            ax.set_title(f"GT {gt}\n(missing file)")
            ax.axis("off")
            continue

        preds = predict_one(model, path, device)
        pred = preds[task]["label"]
        conf = preds[task]["probs"][pred]
        ax.imshow(Image.open(path).convert("RGB"))
        color = "green" if pred == gt else "red"
        ax.set_title(f"GT: {gt}  |  Pred: {pred} ({conf:.2f})", color=color, fontsize=11)
        ax.axis("off")
        rows.append(
            {
                "task": TASK_DISPLAY[task],
                "gt": gt,
                "pred": pred,
                "confidence": round(conf, 3),
                "correct": pred == gt,
                "image": Path(path).name,
            }
        )

    fig.suptitle(TASK_DISPLAY[task], fontsize=14, y=1.02)
    plt.tight_layout()
    out_png = OUT_DIR / f"{task}_examples.png"
    fig.savefig(out_png, dpi=150, bbox_inches="tight")
    print("saved", out_png.name)
    plt.show()
    return pd.DataFrame(rows)


summary_tables = []
for task in TASKS:
    table = run_and_show(task, examples[task])
    if table is not None and len(table):
        summary_tables.append(table)
        display(table)

## Summary

In [ ]:
if summary_tables:
    summary = pd.concat(summary_tables, ignore_index=True)
    display(summary)
    summary.to_csv(OUT_DIR / "summary.csv", index=False)
    print("saved summary.csv")
    if "correct" in summary.columns:
        print("Accuracy on these hand-picked examples:", summary["correct"].mean())
else:
    print("No images found. Check CAPILLAROSCOPY_ROOT / NFC_IMAGE_ROOT on the cluster.")